# VLM QLoRA — Kaggle Resume Training (clean build)

Resumes Stage-2 LoRA from the latest `lora_stepXXXX` checkpoint (we have **3750**).
No pruning. All checkpoints kept. Auto-uploads to your dataset so **you never
hand-download** mid-session.

## One-time setup
1. **Datasets** (right panel -> Add Input) — attach both:
   - `vlm-projector`     -> `projector_stage1.pt`
   - `vlm-session-state` -> `lora_step3750/` + `cls_head.pt` + `faiss_index/` + `stage2.jsonl`
2. **Secrets** (Add-ons -> Secrets) — add all three:
   - `HF_TOKEN`
   - `SEMANTIC_SCHOLAR_API_KEY`
   - `KAGGLE_KEY`  (your `KGAT_...` token, for auto-upload + download)
3. **Settings -> Accelerator -> GPU T4**.
4. **Run All.**

## How saving works (no manual download)
- Cell 8 auto-uploads ALL checkpoints to `vlm-session-state` when training stops.
- Cell 9 = press any time for a mid-session snapshot (does not stop training).
- A hard 12h kill skips the upload, so **interrupt cell 8 yourself before the limit**,
  or let training finish. Fallback copy always sits in `/kaggle/working/session_state/`.


In [ ]:
# 1. CONFIG ===================================================================
import os, re

IS_FIRST_SESSION = False          # resuming from an existing checkpoint

# Auto-detect your Kaggle username from the attached dataset mount path, so the
# SAME notebook works on ANY account with NO editing. Falls back if not found.
def _detect_kaggle_username():
    for root, _dirs, _files in os.walk("/kaggle/input"):
        m = re.search(r"/kaggle/input/datasets/([^/]+)/", root.rstrip("/") + "/")
        if m:
            return m.group(1)
    return None

KAGGLE_USERNAME   = _detect_kaggle_username() or "sujalprasad1"
PROJECTOR_DATASET = "vlm-projector"
STATE_DATASET     = "vlm-session-state"

MAX_PAIRS  = 4000
EPOCHS     = 3
GRAD_ACCUM = 4
SAVE_EVERY = 250
LR         = 2e-4

print("Config:")
print(f"  session     : {'FIRST' if IS_FIRST_SESSION else 'RESUME'}")
print(f"  kaggle user : {KAGGLE_USERNAME}  (auto-detected)")
print(f"  grad_accum  : {GRAD_ACCUM} | max_pairs : {MAX_PAIRS} | save_every : {SAVE_EVERY}")


In [ ]:
# 2. PACKAGES =================================================================
import subprocess, sys
def pip(*a): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *a])
pip("peft>=0.19.1")
pip("bitsandbytes>=0.49.2")
pip("accelerate>=1.13.0")
pip("faiss-cpu==1.13.2")
pip("sentence-transformers")
pip("python-dotenv")
print("Packages ready.")


In [ ]:
# 3. SECRETS & ENV ============================================================
import os, torch
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"]                 = secrets.get_secret("HF_TOKEN")
os.environ["SEMANTIC_SCHOLAR_API_KEY"] = secrets.get_secret("SEMANTIC_SCHOLAR_API_KEY")
os.environ["CUBLAS_WORKSPACE_CONFIG"]  = ":4096:8"
os.environ["MEDDIAG_MAX_VRAM_GB"]      = "14"   # full 16GB T4 (4GB hypothesis confirmed)

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("Secrets loaded, HF login OK. Full 16 GB mode.")


In [ ]:
# 4. CLONE REPO (fresh) =======================================================
import subprocess, os
REPO_URL = "https://github.com/Sreenjoyee/visual-language-model-research-qlora-cot-rag.git"
REPO_DIR = "/kaggle/working/vlm"

if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", "--depth=1", REPO_URL, REPO_DIR])
    print("Cloned ->", REPO_DIR)
else:
    subprocess.call(["git", "-C", REPO_DIR, "reset", "--hard"])
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
    print("Updated ->", REPO_DIR)

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())


In [ ]:
# 5. RESTORE ASSETS (auto-discover under /kaggle/input) =======================
# Mount path varies, so we SEARCH instead of hardcoding. Restores ALL checkpoints.
import shutil, re, os
from pathlib import Path

MODELS_DIR = Path(REPO_DIR) / "models"; MODELS_DIR.mkdir(exist_ok=True)

def find_file(name):
    for root, _d, files in os.walk("/kaggle/input"):
        if name in files: return Path(root) / name
    return None

def find_dir(name):
    for root, dirs, _f in os.walk("/kaggle/input"):
        if name in dirs: return Path(root) / name
    return None

proj = find_file("projector_stage1.pt")
if proj is None:
    raise FileNotFoundError("projector_stage1.pt not found - attach vlm-projector dataset")
shutil.copy2(proj, MODELS_DIR / "projector_stage1.pt")
print(f"projector  ({proj.stat().st_size/1e6:.0f} MB)")

if not IS_FIRST_SESSION:
    fa = find_dir("faiss_index")
    if fa:
        dst = Path(REPO_DIR) / "faiss_index"
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(fa, dst); print("faiss_index restored")
    else:
        print("WARNING: faiss_index not found")

    pat = re.compile(r"lora_step(\d+)$")
    found = []
    for root, dirs, _f in os.walk("/kaggle/input"):
        for d in dirs:
            if pat.match(d): found.append(Path(root) / d)
    if not found:
        raise FileNotFoundError("no lora_step* checkpoint found - attach vlm-session-state dataset")
    for s in found:
        dst = MODELS_DIR / s.name
        if dst.exists(): shutil.rmtree(dst)
        shutil.copytree(s, dst); print("checkpoint restored ->", s.name)

    ch = find_file("cls_head.pt")
    if ch: shutil.copy2(ch, MODELS_DIR / "cls_head.pt"); print("cls_head restored")
    else:  print("WARNING: cls_head.pt not found (classifier will re-converge)")

    lg = find_file("stage2.jsonl")
    if lg:
        (Path(REPO_DIR) / "logs").mkdir(exist_ok=True)
        shutil.copy2(lg, Path(REPO_DIR) / "logs" / "stage2.jsonl"); print("log restored")

print("\nAssets ready.")


In [ ]:
# 6. PIPELINE STATE (skip to Stage-2, clear lock) =============================
from pathlib import Path
sp = Path(REPO_DIR) / "logs" / ".pipeline_state"; sp.parent.mkdir(exist_ok=True)
# mark step0,1,2 done so --resume jumps straight to Stage-2 training (step3)
with open(sp, "w") as f:
    for marker in ["step0"]*7 + ["step1", "step2"]:
        f.write(marker + chr(10))
(Path(REPO_DIR) / "logs" / ".pipeline.lock").unlink(missing_ok=True)
print("Pipeline state set (resume into Stage-2). Lock cleared.")


In [ ]:
# 7. PATCH run_pipeline.sh (grad_accum / max_pairs) ===========================
import re
from pathlib import Path
sh = Path(REPO_DIR) / "run_pipeline.sh"; t = sh.read_text()
t = re.sub(r"(GRAD_ACCUM=)\d+",   f"GRAD_ACCUM={GRAD_ACCUM}",  t)
t = re.sub(r"(MAX_PAIRS_S2=)\d+", f"MAX_PAIRS_S2={MAX_PAIRS}", t)
sh.write_text(t)
print(f"patched: GRAD_ACCUM={GRAD_ACCUM}, MAX_PAIRS_S2={MAX_PAIRS}")


In [ ]:
# 8. VERIFY RESUME, then TRAIN + PERIODIC AUTO-UPLOAD =========================
# Checkpoints are pushed to the vlm-session-state dataset EVERY 30 MIN during
# training (background thread) AND on stop. So they're always downloadable from
# the dataset page, and a hard 12h kill loses at most ~30 min from the dataset
# (and nothing from disk). No pruning — every checkpoint is uploaded.
import subprocess, os, shutil, re, json, threading
from pathlib import Path

MODELS = Path(REPO_DIR) / "models"
pat = re.compile(r"lora_step(\d+)$")
steps = sorted(int(pat.match(p.name).group(1)) for p in MODELS.iterdir() if pat.match(p.name))
if steps:
    top = MODELS / f"lora_step{steps[-1]}"
    ok = (top / "train_state.pt").exists()
    print(f"RESUME TARGET: lora_step{steps[-1]}  (train_state.pt: {'OK' if ok else 'MISSING!'})")
    print(f"all checkpoints present: {steps}")
else:
    print("No checkpoints found - would start from scratch!")
print("=" * 60)

def kaggle_auth():
    try:
        token = secrets.get_secret("KAGGLE_KEY").strip()
    except Exception as e:
        print(f"  !! KAGGLE_KEY secret missing ({e}); skipping upload."); return False
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    kdir = Path.home() / ".kaggle"; kdir.mkdir(exist_ok=True)
    if token.startswith("KGAT_"):
        os.environ["KAGGLE_API_TOKEN"] = token
        f = kdir / "access_token"; f.write_text(token); f.chmod(0o600)
    else:
        os.environ["KAGGLE_KEY"] = token
        f = kdir / "kaggle.json"; f.write_text(json.dumps({"username": KAGGLE_USERNAME, "key": token})); f.chmod(0o600)
    return True

_upload_lock = threading.Lock()

def package_and_upload(msg="auto-update from training"):
    with _upload_lock:
        OUT = Path("/kaggle/working/session_state"); OUT.mkdir(exist_ok=True)
        M = Path(REPO_DIR) / "models"; L = Path(REPO_DIR) / "logs"
        p = re.compile(r"lora_step(\d+)$")
        cps = sorted((int(p.match(x.name).group(1)), x) for x in M.iterdir() if p.match(x.name))
        for _, c in cps:                       # ALL checkpoints — nothing dropped
            d = OUT / c.name
            if d.exists(): shutil.rmtree(d)
            shutil.copytree(c, d)
        print("  packaged:", [c.name for _, c in cps])
        s = M / "cls_head.pt"
        if s.exists(): shutil.copy2(s, OUT / "cls_head.pt")
        fa = Path(REPO_DIR) / "faiss_index"
        if fa.exists():
            d = OUT / "faiss_index"
            if d.exists(): shutil.rmtree(d)
            shutil.copytree(fa, d)
        lg = L / "stage2.jsonl"
        if lg.exists(): shutil.copy2(lg, OUT / "stage2.jsonl")
        if not kaggle_auth():
            print("  fallback: files staged in", OUT); return False
        meta = {"title": STATE_DATASET, "id": f"{KAGGLE_USERNAME}/{STATE_DATASET}", "licenses": [{"name": "CC0-1.0"}]}
        (OUT / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
        r = subprocess.run(["kaggle", "datasets", "version", "-p", str(OUT), "-m", msg, "--dir-mode", "zip"],
                           capture_output=True, text=True)
        good = r.returncode == 0
        print("  [OK] dataset updated" if good else "  [FAIL] " + (r.stderr or r.stdout)[:300])
        return good

# ── background uploader: snapshot to the dataset every 30 min ──────────────────
_stop = threading.Event()
def _periodic():
    last = -1
    while not _stop.wait(1800):   # 1800s = 30 min
        cur = max([int(pat.match(p.name).group(1)) for p in MODELS.iterdir() if pat.match(p.name)] or [-1])
        if cur > last:
            print(f"\n[uploader] periodic snapshot at step {cur} ...")
            if package_and_upload(f"periodic autosave step {cur}"):
                last = cur
threading.Thread(target=_periodic, daemon=True).start()
print("[uploader] periodic auto-upload every 30 min is ON\n")

# ── train ─────────────────────────────────────────────────────────────────────
proc = subprocess.Popen(["bash", "run_pipeline.sh", "--resume"], cwd=REPO_DIR,
                        env=os.environ.copy(), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
try:
    for line in proc.stdout: print(line, end="", flush=True)
except KeyboardInterrupt:
    proc.terminate(); print("\n[notebook] interrupted - saving now...")
proc.wait()
_stop.set()
print(f"\n[notebook] exit code {proc.returncode}")
print("\n[notebook] final packaging + uploading...")
package_and_upload("final upload")
print("\nDownload anytime from: https://www.kaggle.com/datasets/" + KAGGLE_USERNAME + "/" + STATE_DATASET)


In [ ]:
# 9. MANUAL SNAPSHOT (run any time, does NOT stop training) ===================
package_and_upload()


## Downloading the weights

You normally never need to — cell 8 pushes everything to `vlm-session-state`,
and the next session pulls it automatically.

To pull weights to your laptop (stable, unlike the in-session Output tab):
- **Dataset page**: https://www.kaggle.com/datasets/sujalprasad/vlm-session-state -> three-dot menu -> Download
- **CLI**:
  ```
  export KAGGLE_API_TOKEN=KGAT_xxx
  kaggle datasets download sujalprasad/vlm-session-state --unzip -p ./out
  ```

## What was fixed vs the old notebook
- No pruning anywhere — every checkpoint kept and uploaded.
- Resume verified before training (cell 8 prints the exact target step).
- `git reset --hard` before pull — no more merge conflicts.
- Auto-discover dataset paths — mount path can't break it.
- Auto-upload via `KGAT_` token — downloads actually work.
- Kermany pre-load capped at 400 (in repo) — no RAM OOM.
